In [ ]:
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib.ticker import MultipleLocator
from utils import *

In [ ]:
temps = [100,300,500,700,900]
strains = np.linspace(0.99, 1.02, 16)
tmd_formula = {(0,0):'MoS2',(1,0):'WS2',(0,1):'MoSe2',(1,1):'WSe2'}
foldername = 'latparam_md_data/'

fig, axs = plt.subplots(2, 2, figsize=(10, 10),sharex='col')
fontsize = 15
plt.rcParams['font.size'] = fontsize

# Choose a colormap that reflects temperature (inferno, plasma, or hot)
base_cmap = plt.cm.plasma  # Other options: plt.cm.hot, plt.cm.plasma
cmap = mcolors.LinearSegmentedColormap.from_list(
    "custom_inferno", base_cmap(np.linspace(0, 0.7, 256))  # Clip to avoid the lightest colors
)
norm = plt.Normalize(vmin=min(temps), vmax=max(temps))  # Normalize temperatures


for i,(x,y) in enumerate([(0,0),(0,1),(1,0),(1,1)]):
    ax = axs[i // 2, i % 2]  # Determine subplot location
    for temp in temps:
        avg_energies = np.zeros(len(strains))
        for j,strain in enumerate(strains):

            trajname = f'x{x:.3f}y{y:.3f}T{temp:04.0f}_s{strain:5.3f}'
            data = np.load(f'{foldername}/{trajname}.npz')
            energies = data['energies']
            avg_energies[j] = np.mean(energies)
        
        color = cmap(norm(temp))
        alats = strains*a_optB88_QE[x, y]
        ax.scatter(alats,avg_energies,label=f'T = {temp}K',facecolors='none',
                       edgecolors = color, s = 20)
        
        # Fit quadratic curve (2nd-degree polynomial)
        coeffs = np.polyfit(alats, avg_energies, 2)  # ax^2 + bx + c
        poly_fn = np.poly1d(coeffs)  # Create polynomial function
        
        # Find minimum analytically
        a_min = -coeffs[1] / (2 * coeffs[0])  # a_min = -B / 2A
        E_min = poly_fn(a_min)  # Evaluate minimum energy
        
        # Generate smooth points for curve
        alats_smooth = np.linspace(min(alats), max(alats), 100)
        energy_smooth = poly_fn(alats_smooth)
        # Plot quadratic fit
        ax.plot(alats_smooth, energy_smooth, color=color, linestyle='--', alpha=0.8)
        
        # Mark and annotate the minimum point
        ax.scatter(a_min, E_min, color='red', marker='x', s = 100)
        ax.text(a_min, E_min+0.4, rf', $a_{{min}}$  = {a_min:.4f}',
                verticalalignment='bottom', horizontalalignment='center',fontsize=fontsize)
        if i == 0:  # Collect legend items only once
            energy_offset = -9.215e4
            #ax.yaxis.set_major_locator(MultipleLocator(10))  # Set tick spacing
            ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda val, pos: f"{int(val - energy_offset)}"))
            # Add text to indicate the offset (in scientific notation)
            tick_font = ax.yaxis.get_ticklabels()[0].get_fontproperties()
            ax.text(0.0, 1.012, "$-$9.215e4", transform=ax.transAxes,
                    fontproperties=tick_font,fontweight='normal')
            handles, labels = ax.get_legend_handles_labels()
            
    
        if temp == temps[-1]:
            ax.set_ylim(top=max(energy_smooth)+2)
        elif temp == temps[0]:
            ax.set_ylim(bottom=max(energy_smooth)-2)
    
    ax.text(0.02, 0.98, rf"({chr(ord('a')+i)}) {tmd_formula[(x, y)][:-1]}$\bf _{{2}}$",
            va='top',ha='left',fontsize=fontsize+1,transform=ax.transAxes,
           fontweight='bold')
    ax.xaxis.set_minor_locator(MultipleLocator(0.005))

fig.legend(handles, labels, loc='upper center', fontsize=fontsize,
           ncol=len(temps), frameon=False, bbox_to_anchor=(0.5, 0.95),
           prop={'weight': 'bold'},columnspacing=0.1)
fig.text(0.5, 0.06, r"Lattice Parameter, $a$ (in $\AA$)", ha='center',fontsize=fontsize+5)
fig.text(0.043, 0.5, "Mean Trajectory Energy (in eV)", va='center', rotation='vertical',fontsize=fontsize+5)    

plt.subplots_adjust(wspace=0.2, hspace=0.1)

plt.savefig('figures/energy_vs_alat.png',dpi=300,facecolor='white',transparent=False,
            bbox_inches='tight',pad_inches=0.02)

plt.show()